# Lab 08 — VQE: Intuición Guiada

El Variational Quantum Eigensolver (VQE) combina un circuito cuántico parametrizado con un optimizador clásico para encontrar la energía del estado fundamental de un Hamiltoniano.

**Principio variacional**: ⟨ψ(θ)|H|ψ(θ)⟩ ≥ E₀ para cualquier |ψ(θ)⟩

## 1. El Hamiltoniano de ejemplo

Usamos el Hamiltoniano de Heisenberg 1D para 2 spins:

$$H = J(X_0 X_1 + Y_0 Y_1 + Z_0 Z_1)$$

con J=1. Sus autovalores son: E₀ = -3 (estado singlete), E₁ = 1 (tripletes).
Buscamos el estado fundamental con VQE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# Hamiltoniano de Heisenberg 2 spins
H = SparsePauliOp.from_list([
    ('XX', 1.0),
    ('YY', 1.0),
    ('ZZ', 1.0),
])

print('Hamiltoniano:')
print(H)

# Diagonalización exacta para referencia
H_matrix = H.to_matrix()
eigvals = np.linalg.eigvalsh(H_matrix)
print(f'\nAutovalores exactos: {eigvals}')
print(f'Energía fundamental E₀ = {eigvals[0]:.4f}')

## 2. Circuito ansatz

El ansatz define el espacio de búsqueda. Usamos un ansatz hardware-eficiente:
capas de rotaciones Ry parametrizadas + entrelazamiento CNOT.

La profundidad del ansatz controla la expresividad vs el ruido en hardware real.

In [ ]:
def build_ansatz(n_qubits: int = 2, depth: int = 1) -> QuantumCircuit:
    """Ansatz hardware-eficiente con depth capas de Ry + CNOT."""
    n_params = n_qubits * (depth + 1)
    theta = ParameterVector('θ', n_params)
    qc = QuantumCircuit(n_qubits)

    idx = 0
    for layer in range(depth + 1):
        for q in range(n_qubits):
            qc.ry(theta[idx], q)
            idx += 1
        if layer < depth:
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)
    return qc

ansatz = build_ansatz(n_qubits=2, depth=2)
print(ansatz.draw('text'))
print(f'\nParámetros: {ansatz.num_parameters}')

## 3. Función de coste y optimización

La función de coste evalúa la energía esperada ⟨H⟩ para un conjunto de parámetros θ.
Usamos `StatevectorEstimator` para calcular el valor esperado de forma exacta (sin shots).

In [ ]:
estimator = StatevectorEstimator()
energy_history = []

def cost_function(params: np.ndarray) -> float:
    """Calcula ⟨ψ(params)|H|ψ(params)⟩."""
    job = estimator.run([(ansatz, H, params)])
    energy = job.result()[0].data.evs
    energy_history.append(float(energy))
    return float(energy)

# Optimización con COBYLA
np.random.seed(42)
theta0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)

result = minimize(
    cost_function,
    theta0,
    method='COBYLA',
    options={'maxiter': 500, 'rhobeg': 0.5}
)

print(f'Energía encontrada:  {result.fun:.6f}')
print(f'Energía exacta E₀:   {eigvals[0]:.6f}')
print(f'Error:               {abs(result.fun - eigvals[0]):.6f}')
print(f'Iteraciones:         {len(energy_history)}')

## 4. Convergencia del optimizador

La curva de convergencia muestra cómo desciende la energía durante la optimización. Un buen VQE converge de forma monótona hacia E₀.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(energy_history, 'b-', alpha=0.7, linewidth=1.5, label='VQE energía')
ax.axhline(eigvals[0], color='red', linestyle='--', linewidth=2, label=f'E₀ = {eigvals[0]:.3f}')
ax.set_xlabel('Iteración', fontsize=12)
ax.set_ylabel('⟨H⟩ (energía)', fontsize=12)
ax.set_title('Convergencia de VQE — Hamiltoniano de Heisenberg 2 spins', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Paisaje de energía

Para un ansatz de 1 capa con 2 parámetros, podemos visualizar el paisaje de energía completo. Esto ilustra por qué el VQE puede quedar atrapado en mínimos locales (barren plateaus).

In [ ]:
def energy_landscape_2d(theta0_range, theta1_range, n=40):
    """Calcula paisaje de energía para 2 parámetros fijos, resto en 0."""
    ansatz_1d = build_ansatz(2, depth=0)  # 2 params: Ry⊗Ry
    t0s = np.linspace(*theta0_range, n)
    t1s = np.linspace(*theta1_range, n)
    Z = np.zeros((n, n))
    for i, t0 in enumerate(t0s):
        for j, t1 in enumerate(t1s):
            Z[i, j] = cost_function.__wrapped__(np.array([t0, t1])) if hasattr(cost_function, '__wrapped__') else (
                float(estimator.run([(ansatz_1d, H, [t0, t1])]).result()[0].data.evs)
            )
    return t0s, t1s, Z

t0s, t1s, Z = energy_landscape_2d((-np.pi, np.pi), (-np.pi, np.pi), n=30)

fig, ax = plt.subplots(figsize=(7, 6))
c = ax.contourf(t0s, t1s, Z.T, levels=20, cmap='RdBu')
plt.colorbar(c, ax=ax, label='⟨H⟩')
ax.set_xlabel('θ₀', fontsize=12)
ax.set_ylabel('θ₁', fontsize=12)
ax.set_title('Paisaje de energía — ansatz Ry⊗Ry', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Mínimo global en el paisaje: {Z.min():.4f} (E₀ = {eigvals[0]:.4f})')